<a href="https://colab.research.google.com/github/ilfpns/CUDA_study/blob/main/02_CUDA_VectorSum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!nvidia-smi

Mon Mar  9 09:10:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             27W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CPU와 GPU는
- 서로 다른 메모리 영역을 가짐
- 서로 비동기적으로 동시에 실행 가능

CUDA Programming Steps
1. CPU Memory값을 GPU Memory로 가져오기
2. 연산 후 결과를 GPU Memory에서 다시 CPU Memory로 붙여넣기

In [30]:
%%writefile helloCUDA.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define NUM_DATA 1024

__global__ void vecAdd(int *a, int *b, int *c) {
    int idx = threadIdx.x;
    c[idx] = a[idx] + b[idx];
}

int main() {

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t memSize = sizeof(int) * NUM_DATA;

    a = new int[NUM_DATA]; memset(a, 0, memSize);
    b = new int[NUM_DATA]; memset(b, 0, memSize);
    c = new int[NUM_DATA]; memset(c, 0, memSize);

    for (int i = 0; i < NUM_DATA; i++) {
        a[i] = rand() % 10;
        b[i] = rand() % 10;
    }

    cudaMalloc(&d_a, memSize);
    cudaMalloc(&d_b, memSize);
    cudaMalloc(&d_c, memSize);

    cudaMemcpy(d_a, a, memSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, memSize, cudaMemcpyHostToDevice);

    vecAdd<<<1, NUM_DATA>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize(); // device가 끝날 때까지 기다

    cudaMemcpy(c, d_c, memSize, cudaMemcpyDeviceToHost);

    // check results
    bool result = true;
    for (int i = 0; i < NUM_DATA; i++) {
        if ((a[i] + b[i]) != c[i]) {
            printf("[%d] The results is not matched! (%d, %d)\n",
                   i, a[i] + b[i], c[i]);
            result = false;
        }
    }

    if (result)
        printf("GPU works well!\n");

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    delete[] a;
    delete[] b;
    delete[] c;

    return 0;
}

Overwriting helloCUDA.cu


In [31]:
!nvcc helloCUDA.cu -o helloCUDA

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [32]:
!./helloCUDA

GPU works well!
